<a href="https://colab.research.google.com/github/Yukselendincer/datasceinceproject/blob/main/production_risk_guard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# Grafiklerin profesyonel görünmesi ve Türkçe karakter uyumu için stil ayarları
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# =====================================================================
# 1. VERİ SİMÜLASYONU (Üretim, Maliyet ve Puantaj)
# =====================================================================
def generate_mock_data():
    np.random.seed(42)
    dates = pd.date_range(start="2026-01-01", periods=100, freq="D")

    # Üretim & Maliyet Verisi
    production_data = {
        "Tarih": dates,
        "Islenen_Hammadde_Kg": np.random.normal(50000, 2000, 100),
        "Uretilen_Mamul_Kg": np.random.normal(3000, 150, 100),
        "Enerji_Tuketimi_kWh": np.random.normal(12000, 500, 100),
        "Toplam_Maliyet_TL": np.random.normal(250000, 10000, 100),
        "Fire_Orani": np.random.uniform(0.01, 0.04, 100),
    }
    df_prod = pd.DataFrame(production_data)

    # Yapısal iç kontrol anomalileri (saha kaçakları ve giriş hataları simülasyonu)
    df_prod.loc[15, "Enerji_Tuketimi_kWh"] = 18000  # Enerji verimsizliği / Kaçak
    df_prod.loc[45, "Fire_Orani"] = 0.09             # Süreç randıman kaybı / Aşırı fire
    df_prod.loc[75, "Toplam_Maliyet_TL"] = 350000    # Hatalı fatura kaydı / Maliyet şişmesi

    # Puantaj Verisi
    puantaj_data = {
        "Personel_ID": [f"PER_{i:03d}" for i in range(1, 51)],
        "Normal_Mesai_Saat": [180] * 50,
        "Fazla_Mesai_Saat": np.random.exponential(scale=10, size=50).tolist(),
    }
    df_hr = pd.DataFrame(puantaj_data)
    df_hr.loc[12, "Fazla_Mesai_Saat"] = 85           # Puantaj riski / Mevzuata aykırılık

    return df_prod, df_hr

# =====================================================================
# 2. MALİYET VE VERİMLİLİK HESAPLAMALARI
# =====================================================================
def analyze_production_efficiency(df):
    df["Kg_Basi_Maliyet"] = df["Toplam_Maliyet_TL"] / df["Uretilen_Mamul_Kg"]
    df["Kg_Basi_Enerji_kWh"] = df["Enerji_Tuketimi_kWh"] / df["Uretilen_Mamul_Kg"]
    df["Randiman_Yuzde"] = (df["Uretilen_Mamul_Kg"] / df["Islenen_Hammadde_Kg"]) * 100
    return df

# =====================================================================
# 3. YAPAY ZEKA TABANLI ANOMALİ TESPİTİ (Isolation Forest)
# =====================================================================
def detect_operational_anomalies(df):
    features = ["Fire_Orani", "Kg_Basi_Maliyet", "Kg_Basi_Enerji_kWh", "Randiman_Yuzde"]
    X = df[features]

    # Model parametreleri ile çok değişkenli riskleri yakalıyoruz
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    df["Risk_Skoru"] = iso_forest.fit_predict(X)
    df["Inceleme_Gerekli"] = df["Risk_Skoru"].apply(lambda x: "ALARM" if x == -1 else "Normal")
    return df

# =====================================================================
# 4. YÖNETİM DASHBOARD'U VE GÖRSELLEŞTİRME
# =====================================================================
def generate_management_dashboard(df_prod, df_hr):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("İç Kontrol ve Risk Yönetimi Analitik Dashboard", fontsize=18, fontweight='bold', color='#1a365d')

    # Grafik 1: Enerji Tüketim Sapmaları
    sns.lineplot(ax=axes[0, 0], data=df_prod, x="Tarih", y="Kg_Basi_Enerji_kWh", color="#3182ce", label="Enerji Tüketimi (kWh/Kg)")
    alarms = df_prod[df_prod["Inceleme_Gerekli"] == "ALARM"]
    axes[0, 0].scatter(alarms["Tarih"], alarms["Kg_Basi_Enerji_kWh"], color="#e53e3e", s=120, label="Risk Alarmı", zorder=5)
    axes[0, 0].set_title("Kg Başına Enerji Tüketimi ve Süreç Anomalileri", fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel("kWh / Kg")
    axes[0, 0].legend()

    # Grafik 2: Üretim Fire Dağılımı ve Kritik Tolerans Sınırı
    sns.scatterplot(ax=axes[0, 1], data=df_prod, x="Tarih", y="Fire_Orani", hue="Inceleme_Gerekli", palette={"Normal": "#2b6cb0", "ALARM": "#e53e3e"}, s=80)
    axes[0, 1].axhline(y=0.04, color="#dd6b20", linestyle="--", label="Kritik Fire Sınırı (%4)")
    axes[0, 1].set_title("Günlük Üretim Fire Oranları ve Tolerans Aşımı", fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel("Fire Oranı")
    axes[0, 1].legend()

    # Grafik 3: Risk Matrisi (Maliyet vs Randıman Korelasyon Riski)
    sns.scatterplot(ax=axes[1, 0], data=df_prod, x="Randiman_Yuzde", y="Kg_Basi_Maliyet", hue="Inceleme_Gerekli", palette={"Normal": "#2b6cb0", "ALARM": "#e53e3e"}, s=100)
    axes[1, 0].set_title("Kurumsal Risk Matrisi: Randıman & Kg Başı Maliyet İlişkisi", fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel("Üretim Randımanı (%)")
    axes[1, 0].set_ylabel("Kg Başı Maliyet (TL)")

    # Grafik 4: Puantaj ve Suistimal Kontrolü (Fazla Mesai Limitleri)
    top_overtime = df_hr.sort_values(by="Fazla_Mesai_Saat", ascending=False).head(15)
    sns.barplot(ax=axes[1, 1], data=top_overtime, x="Personel_ID", y="Fazla_Mesai_Saat", color="#4a5568")
    axes[1, 1].axhline(y=50, color="#e53e3e", linestyle="-.", label="Aylık Kritik Sınır (50 Saat)")
    axes[1, 1].set_title("Puantaj Riski: En Yüksek Fazla Mesai Yapan 15 Personel", fontsize=12, fontweight='bold')
    axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')
    axes[1, 1].set_ylabel("Fazla Mesai (Saat)")
    axes[1, 1].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # Raporu kaydet
    output_filename = "Uretim_Ic_Kontrol_Analiz_Dashboard.png"
    plt.savefig(output_filename, dpi=150)
    plt.close()
    print(f"✔️ Yönetim raporlama görseli başarıyla üretildi: '{output_filename}'")

# =====================================================================
# ANA ÇALIŞTIRICI SÜREÇ
# =====================================================================
if __name__ == "__main__":
    df_production, df_human_resources = generate_mock_data()
    df_analyzed = analyze_production_efficiency(df_production)
    df_final = detect_operational_anomalies(df_analyzed)

    # Dashboard'u oluştur
    generate_management_dashboard(df_final, df_human_resources)